In [ ]:
using Plots
using Distributed
using LinearAlgebra
using JLD2

num_cores = length(Sys.cpu_info())
if nprocs()==1
    addprocs(num_cores; exeflags=`--project=$(Base.active_project())`)
end

@everywhere begin
    using LatticeAlgorithms
    using LinearAlgebra
    using Dates
    using BlockDiagonals
    using Distributions
end

In [ ]:
println("num_cores = $(num_cores)")

In [ ]:
type_lattice = "qubit_surface"

num_super_samples = 1
num_samples = Int(1e6)
Nv = 1 # Nv has to be 1 for qubit 

dmin, dmax = 3, 7
drange = dmin : 2 : dmax

ϵrange = (10.5 : 0.1 : 11.5)/100

ϵdrange = []
for ϵ in ϵrange
    for d in drange
        push!(ϵdrange, [ϵ, d])
    end
end
println(ϵdrange)


num_samples_each_core = Int(ceil(num_samples/num_cores))
num_total_samples = Int(num_samples_each_core * num_cores);
println([num_samples_each_core, num_samples, num_total_samples])

logfile = "$(type_lattice)_bsv_$(dmin)_$(dmax)_$(ϵrange[1])_$(ϵrange[end])_$(num_total_samples)_log.txt"
    open(logfile, "w") do file
end

In [ ]:
@time results = pmap(1:num_cores) do _
    p_list = Dict(ϵdrange.=>[0.0 for _ in 1 : length(ϵdrange)])
    t_list = Dict(ϵdrange.=>[0.0 for _ in 1 : length(ϵdrange)])
    
    Ms = Dict()
    Ωs = Dict()
    Mperps = Dict()
    invtransposeMqs = Dict()
    invtransposeMperps = Dict()
    transposeΩMperps = Dict()
    for (ind_ϵd, ϵd) in enumerate(ϵdrange)
        ϵ, d = ϵd[1], Int(ϵd[2])
        M = surface_code_M(d) ; 
        Mperp = GKP_logical_operator_generator(M) 
        Ω = Ω_matrix(M)         
        invtransposeMq = inv(transpose(M))[1:2:end, 1:2:end]    
        Ms[d] = M
        Mperps[d] = Mperp
        Ωs[d] = Ω
        invtransposeMqs[d] = invtransposeMq
        invtransposeMperps[d] = inv(√(2π) * transpose(Mperp))
        transposeΩMperps[d] = -transpose(Ω*Mperp)
    end        
    
    for (ind_ϵd, ϵd) in enumerate(ϵdrange)
        ϵ, d = ϵd[1], Int(ϵd[2])
        σ = (2/π * log((1-ϵ)/ϵ))^(-1/2) # ϵ/(1-ϵ) = exp(-π/(2σ^2)) = exp(-(√π)^2/(2σ^2))

        p_I_bsv = 0
        time_bsv = 0        

        ϵdtime = @elapsed for _ in 1 : num_samples_each_core
            ξ = √π * rand(Binomial(1, ϵ), 2d^2)

            if d > 21
                setprecision(BigFloat, 64)
                ξ = BigFloat.(ξ)
            end
            
            ξ2 = -√(2π) * Ms[d] * Ωs[d] * ξ
            s = ξ2 - floor.(ξ2/(2π)) * 2π
            # ηs = -transpose(Ωs[d]*Mperps[d]) * s/√(2π) ; 
            # b = inv(√(2π) * transpose(Mperps[d])) * (ηs-ξ)
            ηs = transposeΩMperps[d] * s/√(2π) ; 
            b = invtransposeMperps[d] * (ηs-ξ)
            @assert norm(round.(Int, b) - b) < 1e-10    
            
            time_bsv += @elapsed rec_q = bsv_surface_code(ηs[1:2:end], σ; Nv=Nv, subspace="x")

            neterror_q = invtransposeMqs[d] * (rec_q+ξ[1:2:end]) / √(2π)                

            norm(round.(Int, neterror_q) - neterror_q) < 1e-10 ? nx = 0 : nx = 1

            if mod(nx, 2) == 0
                p_I_bsv += 1
            end
        end
        p_list[[ϵ, d]] += p_I_bsv
        t_list[[ϵ, d]] += time_bsv

        if myid() == 2 # Print the progress of the 2nd worker
            println(["$(ind_ϵd)/$(length(ϵdrange)), $d, $(ϵdtime), $(string(now()))"])
            open(logfile, "a") do file
                write(file, "$(ind_ϵd)/$(length(ϵdrange)), $d, $(ϵdtime), $(string(now()))\n")
            end
        end
    end
    
    return p_list, t_list
end ;     

t_list = merge(+, [res[2] for res in results]...) 
p_list = merge(+, [res[1] for res in results]...)


map!(v->v./num_total_samples, values(t_list))
map!(v->v./num_total_samples, values(p_list))

# Save the result
fn = "$(type_lattice)_bsv_$(dmin)_$(dmax)_$(ϵrange[1])_$(ϵrange[end])_$(num_total_samples).jld2";
jldsave(fn; 
    ϵrange=ϵrange, 
    num_samples=num_samples_each_core*num_cores,
    p_list = p_list,
    t_list = t_list,
    drange = drange
)

In [ ]:
sort(load(fn)["p_list"])

# Compare to existing data

In [ ]:
function get_p0list_sorted(p_list, drange, ϵrange)
    p0list_sorted = sort(p_list)
    p0list_sorted = collect(values(p0list_sorted))
    p0list_sorted = reshape(p0list_sorted, (length(drange), length(ϵrange)))
    p0list_sorted = [p0list_sorted[:,i] for i in 1:size(p0list_sorted,2)]
    return p0list_sorted
end

In [ ]:
new_data = sort(load(fn))
new_p_list = new_data["p_list"]
new_p_list_sorted = get_p0list_sorted(new_p_list, drange, ϵrange)

In [ ]:
old_data = sort(load("data/qubit_surface_bsv_3_15_0.105_0.115_1000032.jld2"))
old_p_list = Dict()
for (k, v) in old_data["p_list"]
    if k[2] ∈ drange # && k[1] ∈ ϵrange
        old_p_list[k] = v
    end
end

old_p_list_sorted = get_p0list_sorted(old_p_list, drange, old_data["ϵrange"])

In [ ]:
linecolors = get_color_palette(:auto, plot_color(:white))

g = plot()
for (ind_d, d) in enumerate(drange)
    new_ps = [item[ind_d] for item in new_p_list_sorted]
    old_ps = [item[ind_d] for item in old_p_list_sorted]
    yerrnew = sqrt.(new_ps .* (1 .- new_ps) ./ num_total_samples)
    yerrold = sqrt.(old_ps .* (1 .- old_ps) ./ 1e6)
    plot!(ϵrange, new_ps, label="new data, d=$d", marker=:circle, color=linecolors[ind_d], yerr=yerrnew)
    println()
    plot!(old_data["ϵrange"], old_ps, label="old data, d=$d", marker=:star, color=linecolors[ind_d], yerr=yerrold)
end
plot!(xlabel="ϵ", ylabel="fidelity", size=(1200, 400))

    